# Trực quan hóa Lịch sử Huấn luyện - EfficientNet - B0

Notebook này phân tích và vẽ biểu đồ quá trình huấn luyện từ file `history_spatial.json`:
- **Loss Curve**: Train Loss vs Val Loss theo từng epoch
- **AUC Curve**: Train AUC vs Val AUC (ROC-AUC)
- **Balanced Accuracy Curve**: Train Bal-ACC vs Val Bal-ACC
- **Biểu đồ tổng hợp (Dashboard)**: Độ phân giải cao (300 DPI) sẵn sàng dùng cho báo cáo / khóa luận tốt nghiệp.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thiết lập font và style đồ thị học thuật
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.35
plt.rcParams['grid.linestyle'] = '--'
plt.rcParams['axes.edgecolor'] = '#bbbbbb'
plt.rcParams['axes.linewidth'] = 0.8

## 1. Nạp và Xử lý dữ liệu từ `history_spatial.json`

In [ ]:
# Tìm đường dẫn file JSON tương đối hoặc tuyệt đối
json_path = os.path.abspath(os.path.join("..", "..", "code", "system", "checkpoints", "spatial_baseline", "history_spatial.json"))
if not os.path.exists(json_path):
    json_path = r"D:\nam_4\DATN\code\system\checkpoints\spatial_baseline\history_spatial.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

history = data["history"]
best_epoch = data.get("best_epoch", 0) + 1  # Đổi sang 1-indexed (Epoch 1 -> 16)
best_val = data.get("best_val_metrics", {})

print("=" * 60)
print(f"-> Run Name        : {data.get('config', {}).get('run_name')}")
print(f"-> Tổng số Epochs  : {len(history)}")
print(f"-> Best Epoch      : {best_epoch} (Loss: {data.get('final_val_loss', 0):.4f})")
print(f"-> Best Val AUC    : {best_val.get('auc', 0):.4f}")
print(f"-> Best Val Bal-ACC: {best_val.get('balanced_acc', 0):.4f}")
print("=" * 60)

# Chuyển đổi toàn bộ metrics theo epoch thành DataFrame
df = pd.DataFrame([{
    "epoch": h["epoch"] + 1,
    "train_loss": h["train_loss"],
    "val_loss": h["val_loss"],
    "train_auc": h["train_auc"],
    "val_auc": h["val_auc"],
    "train_balanced_acc": h.get("train_balanced_acc"),
    "val_balanced_acc": h.get("val_balanced_acc"),
    "val_ap": h.get("val_ap"),
    "is_best": h.get("is_best", False)
} for h in history])

display(df)

## 2. Biểu đồ tổng hợp 3 trong 1 (Loss, AUC, Balanced Accuracy)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.2))
epochs = df["epoch"]

# Subplot 1: Train Loss vs Val Loss
ax1 = axes[0]
ax1.plot(epochs, df["train_loss"], marker="o", markersize=5, color="#1f77b4", label="Train Loss", linewidth=2)
ax1.plot(epochs, df["val_loss"], marker="s", markersize=5, color="#d62728", label="Val Loss", linewidth=2)
ax1.axvline(best_epoch, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Best Epoch ({best_epoch})")
ax1.set_title("Train vs Val Loss", fontsize=13, fontweight="bold", pad=12)
ax1.set_xlabel("Epoch", fontsize=11)
ax1.set_ylabel("Loss", fontsize=11)
ax1.set_xticks(epochs)
ax1.legend(frameon=True, facecolor="white", edgecolor="#dddddd", shadow=False)

# Subplot 2: Train AUC vs Val AUC
ax2 = axes[1]
ax2.plot(epochs, df["train_auc"], marker="o", markersize=5, color="#1f77b4", label="Train AUC", linewidth=2)
ax2.plot(epochs, df["val_auc"], marker="s", markersize=5, color="#2ca02c", label="Val AUC", linewidth=2)
ax2.axvline(best_epoch, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Best Epoch ({best_epoch})")
ax2.set_title("Train vs Val AUC (ROC-AUC)", fontsize=13, fontweight="bold", pad=12)
ax2.set_xlabel("Epoch", fontsize=11)
ax2.set_ylabel("AUC", fontsize=11)
ax2.set_ylim(0.84, 1.005)
ax2.set_xticks(epochs)
ax2.legend(frameon=True, facecolor="white", edgecolor="#dddddd", shadow=False, loc="lower right")

# Subplot 3: Train Balanced Accuracy vs Val Balanced Accuracy
ax3 = axes[2]
ax3.plot(epochs, df["train_balanced_acc"], marker="o", markersize=5, color="#1f77b4", label="Train Balanced Accuracy", linewidth=2)
ax3.plot(epochs, df["val_balanced_acc"], marker="s", markersize=5, color="#ff7f0e", label="Val Balanced Accuracy", linewidth=2)
ax3.axvline(best_epoch, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Best Epoch ({best_epoch})")
ax3.set_title("Train vs Val Balanced Accuracy", fontsize=13, fontweight="bold", pad=12)
ax3.set_xlabel("Epoch", fontsize=11)
ax3.set_ylabel("Balanced Accuracy", fontsize=11)
ax3.set_ylim(0.76, 1.005)
ax3.set_xticks(epochs)
ax3.legend(frameon=True, facecolor="white", edgecolor="#dddddd", shadow=False, loc="lower right")

plt.suptitle("EfficientNet - B0 - Tiến trình Huấn luyện & Đánh giá", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()

# Lưu ảnh chất lượng cao 300 DPI
output_summary_path = os.path.join(".", "spatial_baseline_dashboard.png")
plt.savefig(output_summary_path, dpi=300, bbox_inches="tight")
print(f"-> Đã lưu ảnh tổng hợp tại: {os.path.abspath(output_summary_path)}")
plt.show()

## 3. Biểu đồ riêng từng tiêu chí (Chuẩn bị hình ảnh cho Khóa Luận/Báo Cáo)

In [ ]:
# 3.1 Biểu đồ Loss riêng biệt
plt.figure(figsize=(8, 5))
plt.plot(epochs, df["train_loss"], marker="o", markersize=6, color="#1f77b4", label="Train Loss", linewidth=2.2)
plt.plot(epochs, df["val_loss"], marker="s", markersize=6, color="#d62728", label="Val Loss", linewidth=2.2)
plt.axvline(best_epoch, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Best Epoch ({best_epoch})")

# Đánh dấu giá trị val loss thấp nhất
min_val_loss = df["val_loss"].min()
min_epoch = df.loc[df["val_loss"].idxmin(), "epoch"]
plt.annotate(f"Min Val Loss: {min_val_loss:.4f}", 
             xy=(min_epoch, min_val_loss), 
             xytext=(min_epoch - 3.5, min_val_loss + 0.08),
             arrowprops=dict(facecolor="black", shrink=0.08, width=1, headwidth=6),
             fontsize=10, fontweight="bold", bbox=dict(boxstyle="round,pad=0.3", fc="yellow", alpha=0.3))

plt.title("EfficientNet - B0 - Tiến trình Hội tụ Hàm mất mát (Loss Curve)", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Loss", fontsize=11)
plt.xticks(epochs)
plt.legend(frameon=True, facecolor="white", edgecolor="#dddddd", shadow=False)
plt.tight_layout()
plt.savefig("spatial_loss_curve.png", dpi=300, bbox_inches="tight")
print("-> Đã lưu ảnh: spatial_loss_curve.png")
plt.show()

In [ ]:
# 3.2 Biểu đồ AUC riêng biệt
plt.figure(figsize=(8, 5))
plt.plot(epochs, df["val_auc"], marker="s", markersize=6, color="#2ca02c", label="Validation AUC", linewidth=2.2)
plt.plot(epochs, df["train_auc"], marker="o", markersize=5, linestyle="--", color="#1f77b4", alpha=0.7, label="Train AUC", linewidth=1.6)

plt.axvline(best_epoch, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Best Epoch ({best_epoch})")
plt.title("EfficientNet - B0 - Đánh giá AUC", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("ROC-AUC", fontsize=11)
plt.ylim(0.84, 1.005)
plt.xticks(epochs)
plt.legend(frameon=True, facecolor="white", edgecolor="#dddddd", shadow=False, loc="lower right")
plt.tight_layout()
plt.savefig("spatial_auc_curve.png", dpi=300, bbox_inches="tight")
print("-> Đã lưu ảnh: spatial_auc_curve.png")
plt.show()

In [ ]:
# 3.3 Biểu đồ Balanced Accuracy riêng biệt
plt.figure(figsize=(8, 5))
plt.plot(epochs, df["val_balanced_acc"], marker="^", markersize=6, color="#ff7f0e", label="Validation Balanced Accuracy", linewidth=2.2)
plt.plot(epochs, df["train_balanced_acc"], marker="x", markersize=5, linestyle="--", color="#9467bd", alpha=0.7, label="Train Balanced Accuracy", linewidth=1.6)

plt.axvline(best_epoch, color="#2ca02c", linestyle=":", linewidth=1.8, label=f"Best Epoch ({best_epoch})")
plt.title("EfficientNet - B0 - Đánh giá Balanced Accuracy", fontsize=13, fontweight="bold", pad=12)
plt.xlabel("Epoch", fontsize=11)
plt.ylabel("Balanced Accuracy", fontsize=11)
plt.ylim(0.76, 1.01)
plt.xticks(epochs)
plt.legend(frameon=True, facecolor="white", edgecolor="#dddddd", shadow=False, loc="lower right")
plt.tight_layout()
plt.savefig("spatial_balacc_curve.png", dpi=300, bbox_inches="tight")
print("-> Đã lưu ảnh: spatial_balacc_curve.png")
plt.show()